In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


import copy

import gymnasium as gym
import matplotlib.pyplot as plt

from torch.utils.tensorboard import SummaryWriter

# import sys, os
# sys.path.append(os.getcwd()+"/../src")
import jax
from mcts_torch import MCTSNode, MCTSTree

In [2]:
class RepresentationModel(nn.Module): # s0 = h(past_observations)
    def __init__(self, dim_obs_in, n_hidden, n_hidden_state): # TODO encode several past observations, TODO encode actions
        super().__init__()
        
        self.dim_obs_in = dim_obs_in

        self.hidden_state_net = nn.Sequential(
            nn.Linear(dim_obs_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden_state),
        )

    def forward(self, obs): # TODO encode action in input to dynamics function
        if not isinstance(obs, torch.Tensor): obs = torch.tensor(obs)
        hidden_state = self.hidden_state_net(obs)
        return hidden_state

class PredictionModel(nn.Module): # p_k, v_k = f(hidden)
    def __init__(self, n_hidden_state, n_actions):
        super().__init__()

        # self.prediction_net_body = nn.Sequential(
        #     nn.
        # )

        self.prediction_net_policy_head = nn.Sequential(
            nn.Linear(in_features=n_hidden_state, out_features=n_actions)
        )

        self.prediction_net_value_head = nn.Sequential(
            nn.Linear(in_features=n_hidden_state, out_features=1)
        )

    def forward(self, hidden_state):
        if not isinstance(hidden_state, torch.Tensor): hidden_state = torch.tensor(hidden_state)

        #_prediction_body   = self.prediction_net_body(hidden_state)
        _prediction_body = hidden_state
        policy_logits_pred = self.prediction_net_policy_head(_prediction_body)
        value_logits_pred  = self.prediction_net_value_head(_prediction_body)
        return policy_logits_pred, value_logits_pred

class DynamicsModel(nn.Module): # r_k, s_k = g(skm1, a_k) # reward, new hidden state
    def __init__(self, n_hidden_state, n_actions):
        super().__init__()
        self.n_actions = n_actions
        # rk, sk = g(skm1, ak) # reward, new hidden state
        # TODO encode action
        self.dynamics_net_body = nn.Sequential(
            nn.Linear(in_features=n_hidden_state+1, out_features=n_hidden_state) # down-sampling
        )

        self.dynamics_net_reward_head = nn.Sequential(
            nn.Linear(in_features=n_hidden_state, out_features=1)
        )

        self.dynamics_net_next_state_head = nn.Sequential(
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
            nn.ReLU(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
        )

    def forward(self, hidden_state, action): # discrete action -> transform to [0, 1] by dividing by action space size
        if not isinstance(hidden_state, torch.Tensor): hidden_state = torch.tensor(hidden_state)
        if not isinstance(action, torch.Tensor): action = torch.tensor(action)
        if hidden_state.ndim == action.ndim+1: action = action.unsqueeze(action.ndim)
        hidden_state = torch.cat([hidden_state, action], dim=hidden_state.ndim-1)
        _dynamics_body   = self.dynamics_net_body(hidden_state)
        reward_logits_pred = self.dynamics_net_reward_head(_dynamics_body)
        next_state_pred  = self.dynamics_net_next_state_head(_dynamics_body)
        return reward_logits_pred, next_state_pred

In [3]:
# # without batch dim
# s = torch.rand(6)
# a = torch.tensor([3 / 10])

# # want shape 7
# print(s)
# print(torch.cat([s, a], dim=s.ndim-1))

# # with batch
# s_batch = torch.rand((2, 6))
# a_batch = torch.tensor([[3 / 10], [4/10]])
# print(a_batch.shape)

# print(s_batch)
# print(torch.cat([s_batch, a_batch], dim=s_batch.ndim-1))


In [4]:
# n_hidden_state = 6
# n_actions=8
# h = RepresentationModel(dim_obs_in=8, n_hidden=16, n_hidden_state=n_hidden_state)
# f = PredictionModel(n_hidden_state, n_actions=n_actions)
# g = DynamicsModel(n_hidden_state=n_hidden_state, n_actions=n_actions)

# o = torch.rand((10, 8))
# h(o).shape

# s = torch.rand((10, 6))
# f(s)[0].shape, f(s)[1].shape

# a_batch = torch.multinomial(torch.ones(10), num_samples=10, replacement=True).reshape((10,1))
# g(s, a_batch)[0].shape, g(s, a_batch)[1].shape

In [ ]:
# Setup

env_name = "CartPole-v1"
env = gym.make(env_name)


gamma = 0.997
alpha = 0.1 # TODO this is not given in the paper -> look up in pseudocode
buffer_size = 1000
mini_batch_size = 32
n_train_iterations = 100 # how many mini batches are sampled from the buffer

n_actions = env.action_space.n
n_hidden_state = 6

h = RepresentationModel(dim_obs_in=env.observation_space.shape[0], n_hidden=16, n_hidden_state=n_hidden_state)
f = PredictionModel(n_hidden_state, n_actions=n_actions)
g = DynamicsModel(n_hidden_state=n_hidden_state, n_actions=n_actions)

optimizer = torch.optim.Adam(params=list(h.parameters()) + list(f.parameters()) + list(g.parameters()), lr=alpha)

In [6]:
buffer_S          = torch.empty(size=(buffer_size, *env.observation_space.shape), dtype=torch.float32)
buffer_A          = torch.empty(size=(buffer_size,), dtype=torch.int32)
buffer_R          = torch.empty(size=(buffer_size,), dtype=torch.float32)
buffer_S_prime    = torch.empty(size=(buffer_size, *env.observation_space.shape), dtype=torch.float32)
buffer_terminated = torch.empty(size=(buffer_size,), dtype=torch.bool)
buffer_truncated  = torch.empty(size=(buffer_size,), dtype=torch.bool)
buffer_nu         = torch.empty(size=(buffer_size,), dtype=torch.float32)
buffer_pi         = torch.empty(size=(buffer_size, n_actions), dtype=torch.float32)

In [7]:
# Generate experience with saved data_generation_model
# Refresh entire buffer

h_checkpoint = copy.deepcopy(h)
f_checkpoint = copy.deepcopy(f)
g_checkpoint = copy.deepcopy(g)
for model in [h_checkpoint, f_checkpoint, g_checkpoint]:
    for param in model.parameters():
        param.requires_grad = False

S, info = env.reset()
done = False

# TODO this should be train=False, same for MCTS
for istep in range(buffer_size):

    # here, action selection is random sampling according to MCTS-improved policy + noise
    # but we don't do MCTS for now, so it's just sample + noise
    # torch.multinomial(pk, num_samples=1)
    with torch.no_grad():
        hidden_state                          = h_checkpoint(torch.tensor(S, dtype=torch.float32))
        policy_logits_pred, value_logits_pred = f_checkpoint(hidden_state)
        A = torch.multinomial(F.softmax(policy_logits_pred, dim=-1), 1)
        # TODO should you also use MCTS for filling the buffer? I guess yes, but for Atari it didn't matter much
        reward_logits_pred, next_state_pred   = g_checkpoint(hidden_state, A)

    S_prime, R, terminated, truncated, info = env.step(A.item())

    # Store in buffer
    # for now, save states, actions, rewards, next states, truncated/terminated
    # Calculate n-step returns later
    # Re-calculate or update save method for any values of the play_network
    buffer_S[istep, ...]          = torch.tensor(S, dtype=buffer_S.dtype)
    buffer_A[istep, ...]          = A.to(dtype=buffer_A.dtype)
    buffer_R[istep, ...]          = torch.tensor(R, dtype=buffer_R.dtype)
    buffer_S_prime[istep, ...]    = torch.tensor(S_prime, dtype=buffer_S_prime.dtype)
    buffer_terminated[istep, ...] = torch.tensor(terminated, dtype=buffer_terminated.dtype)
    buffer_truncated[istep, ...]  = torch.tensor(truncated, dtype=buffer_truncated.dtype)
    

    done = terminated or truncated

    if done:
        S, info = env.reset()
        done = False
    else:
        S = S_prime

In [8]:
buffer_S_prime.requires_grad

False

In [9]:
# Calculate some things on the buffer...
# MCTS on every observation in the buffer: improved policy, improved value (nu, pi)
seed = 42
rng_key = jax.random.PRNGKey(seed)

# TODO: rather than for istep, do batch everything

for istep in range(buffer_size):
    with torch.no_grad():
        S = buffer_S[istep, ...]
        A = buffer_A[istep, ...]
        R = buffer_R[istep, ...]
        S_prime = buffer_S_prime[istep, ...]
        terminated = buffer_terminated[istep, ...]
        truncated = buffer_truncated[istep, ...]

        hidden_state_root = h(S)
        p_root, v_root = f(hidden_state_root)

        tree = MCTSTree(
            MCTSNode(
                hidden_state=hidden_state_root,
                n_actions=n_actions,
                v = v_root,
                identifier="root",
                ),
            n_simulations=10,
            dynamics_function=g,
            prediction_function=f,
            n_actions=n_actions,
            rng_key=rng_key
        )
        tree.search()
        nu = tree.root_node.v # use in n-step bootstrap target
        pi = tree.root_node.N/tree.root_node.N.sum() # use as policy network target

        buffer_nu[istep, ...] = nu
        buffer_pi[istep, ...] = pi

In [10]:
# TODO weighted integer representatation of reward and value)
# TODO scaling of targets with invertible transformation h (page 14)
# Reward loss (cross entropy)
def l_r(input, target):
    return torch.sqrt(torch.mean(torch.square(input - target), dim=1))

# Value loss (cross entropy)
def l_v(input, target):
    return torch.sqrt(torch.mean(torch.square(input - target), dim=1))

# Policy loss (cross entropy)
def l_p(input_logits, target_probs):
    """Policy loss function"""
    return nn.CrossEntropyLoss(reduction='none')(input_logits, target_probs)

# Test 
# input_logits = torch.randn((4, 6))
# target_logits = torch.randn((4, 6))
# print(l_p(input_logits, F.softmax(target_logits, dim=1)))

# # This is what I want: pi^T \cdot log p_vector , pi is the target
# # however, it loses a dimension, is not [4, 1], but [4]
# print(- (F.softmax(target_logits, dim=1) * (F.log_softmax(input_logits, dim=1))).sum(dim=1))

# l_r(input_logits, target_logits).shape

In [ ]:
# sample a mini batch, calc and train
# TODO prioritized sampling

for i_iteration in range(n_train_iterations):
    ixs = np.random.default_rng().choice(buffer_size-1, mini_batch_size, replace=False)

    mb_S = buffer_S[ixs, ...]
    mb_A = buffer_A[ixs, ...]
    mb_R = buffer_R[ixs, ...]
    mb_S_prime = buffer_S_prime[ixs, ...]
    mb_terminated = buffer_terminated[ixs, ...]
    mb_truncated = buffer_truncated[ixs, ...]
    mb_nu = buffer_nu[ixs, ...]
    mb_pi = buffer_pi[ixs, ...]

    # n-step bootstap target, but with n-step value from output of MCTS search
    # n=1 TODO n=10
    # S, A, R, S_prime, terminated, truncated
    # z = u + gamma*improved_value # TODO n-step bootstrap
    mb_terminal = torch.logical_or(mb_terminated, mb_truncated)
    mb_z = mb_R  + (gamma * mb_nu) * (1-1*mb_terminal)


    # Reevaluated model: hidden_state, policy_logits_pred, value_logits_pred, reward_logits_pred, next_state_pred
    # TODO: this is pseudo code: this should be a sum over k, K successive steps, evaluations of the current model along 5 steps of generated data
    # TODO add parameter norm penalty
    # TODO rename R to u, and reward predictions to r

    # k=1
    mb_hidden_state = h(mb_S)
    mb_p_pred1, mb_v_pred1 = f(mb_hidden_state)
    # check code: mb_A = torch.multinomial(F.softmax(policy_logits_pred, dim=-1), 1) # TODO should these be the actions taken during data generation?
    mb_R_pred1, mb_S_prime1 = g(mb_hidden_state, mb_A)

    # k=2
    # need these from the buffer
    mb_S2 = buffer_S[ixs+1, ...]
    mb_A2 = buffer_A[ixs+1, ...]
    mb_R2 = buffer_R[ixs+1, ...]
    mb_S_prime2 = buffer_S_prime[ixs+1, ...]
    mb_terminated2 = buffer_terminated[ixs+1, ...]
    mb_truncated2 = buffer_truncated[ixs+1, ...]
    mb_nu2 = buffer_nu[ixs+1, ...]
    mb_pi2 = buffer_pi[ixs+1, ...]
    
    mb_p_pred2, mb_v_pred2 = f(mb_S_prime1)
    mb_R_pred2, mb_S_prime2 = g(mb_S_prime1, mb_A2)

    # need to compute this
    mb_terminal2 = torch.logical_or(mb_terminated2, mb_truncated2)
    mb_z2 = mb_R2  + (gamma * mb_nu2) * (1-1*mb_terminal2)

    mb_loss1 = l_r(mb_R_pred1, mb_R) + l_v(mb_v_pred1, mb_z) + l_p(mb_p_pred1, mb_pi)
    mb_loss2 = l_r(mb_R_pred2, mb_R2) + l_v(mb_v_pred2, mb_z2) + l_p(mb_p_pred2, mb_pi2)
    
    loss = 0.5*(mb_loss1.mean() + mb_loss2.mean())

    # Train one step
    for model in [h, f, g]:
        for param in model.parameters():
            param.grad = None
    loss.backward()
    optimizer.step()

In [14]:
# Evaluation
use_MCTS = False

S, info = env.reset()
done = False
total_reward = 0

while not done:
    hidden_state = h(torch.tensor(S, dtype=torch.float32))
    policy_logits_pred, value_logits_pred = f(hidden_state)
    
    if use_MCTS:
        pass
    else:
        A = torch.multinomial(F.softmax(policy_logits_pred, dim=-1), 1)
    
    reward_logits_pred, next_state_pred = g(hidden_state, A)

    S_prime, R, terminated, truncated, info = env.step(A.item())
    total_reward += R
    done = terminated or truncated

print(total_reward)

19.0
